# The cheatsheet as a machine-readable guideline, and its link with O'FAIRe

The quick reference table of the Metadata Guidelines (the cheatsheet in `docs/`) lists the 99 metadata
attributes with their usage level (Mandatory, Recommended, Optional), their FAIR principles and their
property, with the typos of the draft corrected. This notebook:

- **I.** reads the cheatsheet into a table;
- **II.** finds where EarthPortal really stores each attribute;
- **III.** saves the result as the **machine-readable guideline** (`mapping/guideline_profile.csv`);
- **IV.** links each O'FAIRe question to the attributes it reads (`mapping/ofaire_vs_guideline.csv`).

In [1]:
# Imports
import json
import re

import pandas as pd

from config import *

# I. Reading the cheatsheet

In [2]:
# Each row of the cheatsheet table looks like:   1.7 & Status & M & R & A2 & \propb{omv:status} \\
# Section titles look like:                       \textbf{\color{dtnavy}1. General}
ROW = re.compile(r"^\s*(\d+\.\d+) & (.+?) & ([MRO]) & (.+?) & (.+?) & \\propb\{([^}]+)\}")
SECTION = re.compile(r"\\color\{dtnavy\}\d+\. ([^}]+)\}")
USAGE = {"M": "Mandatory", "R": "Recommended", "O": "Optional"}

entries, section = [], ""
for line in CHEATSHEET.read_text(encoding="utf-8").splitlines():
    if (s := SECTION.search(line)):
        section = s.group(1)
    elif (r := ROW.match(line)):
        entries.append({"id": r.group(1), "section": section, "entry": r.group(2),
                        "usage": USAGE[r.group(3)], "fsf_profile": r.group(4).replace("--", ""),
                        "fair": re.sub(r"\\added\{([^}]*)\}", r"\1", r.group(5)).replace("--", ""),
                        "property": r.group(6).replace("dcterms:", "dct:")})
guideline = pd.DataFrame(entries)
print(f"{len(guideline)} attributes read from the cheatsheet")
guideline.usage.value_counts()

99 attributes read from the cheatsheet


usage
Recommended    58
Optional       21
Mandatory      20
Name: count, dtype: int64

# II. Where EarthPortal stores each attribute

EarthPortal stores its metadata under its own property names, mostly from MOD 1.4. An attribute of the
cheatsheet finds its portal field by the first rule that works:

1. **same**: the cheatsheet property is the portal's own property;
2. **mapping**: the portal itself lists the cheatsheet property as an equivalent (its `metadataMappings`);
3. **manual**: a short, justified list in `reference/manual_links.csv`.

In [3]:
# We read the portal's schema saved by notebook 01. Fields without a namespace belong to the
# portal's own namespace, written bpm: (as O'FAIRe and the guideline do)
schema = json.load(open(RAW / "metadata_schema.json", encoding="utf-8"))
fields, mappings = {}, {}
for record in ("submission", "ontology"):
    for f in schema[record]:
        name = f"{f.get('namespace') or 'bpm'}:{f['attribute']}"
        fields.setdefault(name, (record, f))
        for equivalent in f.get("metadataMappings") or []:
            mappings.setdefault(equivalent.replace("dcterms:", "dct:"), (record, name))

manual = pd.read_csv(MANUAL_LINKS).set_index("entry")
manual

,portal_object,portal_attribute,reason
entry,,,
Acronym,ontology,omv:acronym,The cheatsheet names mod:acronym (MOD 3.0); th...
Categories,ontology,omv:hasDomain,omv:hasDomain exists on both the ontology reco...
Version IRI,submission,owl:versionIRI,The cheatsheet names omv:versionIRI; the porta...
This ontology is a view of,ontology,bpm:viewOf,The cheatsheet names dct:isPartOf; the portal ...
Generally related to,submission,door:ontologyRelatedTo,The cheatsheet names mod:semanticArtefactRelat...
Disparate modelling with,submission,door:hasDisparateModelling,The cheatsheet names mod:hasDisparateModelling...


In [4]:
def portal_field(row):
    if row.entry in manual.index:
        return manual.loc[row.entry, "portal_object"], manual.loc[row.entry, "portal_attribute"], "manual"
    if row.property in fields:
        return fields[row.property][0], row.property, "same"
    if row.property in mappings:
        return mappings[row.property][0], mappings[row.property][1], "mapping"
    return "", "", "not found"

guideline[["portal_object", "portal_attribute", "link_rule"]] = guideline.apply(portal_field, axis=1, result_type="expand")
guideline["api_key"] = guideline.portal_attribute.str.split(":").str[1]
guideline["differs_from_portal"] = (guideline.property != guideline.portal_attribute).map({True: "yes", False: "no"})
print(guideline.link_rule.value_counts().to_dict())
print(f"{(guideline.differs_from_portal == 'yes').sum()} attributes are stored by the portal under another property")
guideline[guideline.differs_from_portal == "yes"][["id", "entry", "usage", "property", "portal_attribute", "link_rule"]]

{'same': 72, 'mapping': 21, 'manual': 6}
26 attributes are stored by the portal under another property


,id,entry,usage,property,portal_attribute,link_rule
0,1.1,Acronym,Mandatory,mod:acronym,omv:acronym,manual
8,1.9,Version IRI,Recommended,omv:versionIRI,owl:versionIRI,manual
19,2.6,Keywords,Recommended,dcat:keyword,omv:keywords,mapping
20,2.7,Subjects,Recommended,dct:subject,omv:hasDomain,mapping
25,3.2,Modification date,Mandatory,dct:modified,omv:modificationDate,mapping
28,3.5,Submission date,Mandatory,dct:dateSubmitted,omv:creationDate,mapping
35,5.2,Creator,Mandatory,dct:creator,omv:hasCreator,mapping
36,5.3,Contributor,Recommended,dct:contributor,omv:hasContributor,mapping
40,5.7,Endorsed by,Recommended,mod:endorsedBy,omv:endorsedBy,mapping
55,8.1,Bug database,Recommended,doap:bug-database,doap:bugDatabase,mapping


# III. The machine-readable guideline

We add two pieces of information that a program needs: what kind of value the portal expects (an IRI,
a list, a date...) with its allowed values, and **who can fill the field in**. The last one is a
proposal, to be validated with the team:

- **portal**: the portal generates the value itself (download links, search endpoints, submission date);
- **curator**: the EarthPortal team sets it (categories, groups, visibility, administrators, curation);
- **author**: everything else, filled in the file or in the submission form by the producer.

In [5]:
PORTAL = {"bpm:csvDump", "void:dataDump", "void:uriLookupEndpoint", "void:openSearchDescription", "omv:creationDate"}
CURATOR = {"bpm:administeredBy", "bpm:group", "bpm:viewingRestriction", "bpm:viewOf",
           "schema:includedInDataCatalog", "pav:curatedBy", "pav:curatedOn"}

def who_fixes(row):
    if row.portal_attribute in PORTAL:
        return "portal"
    if row.portal_attribute in CURATOR or (row.entry == "Categories"):
        return "curator"
    return "author"

def expects(row):
    f = fields.get(row.portal_attribute, (None, {}))[1] if row.portal_object else {}
    for record in ("submission", "ontology"):        # the same name can exist on both records
        for g in schema[record]:
            if record == row.portal_object and f"{g.get('namespace') or 'bpm'}:{g['attribute']}" == row.portal_attribute:
                f = g
    allowed = f.get("enforcedValues") or []
    allowed = list(allowed) if isinstance(allowed, dict) else allowed
    return ", ".join(e for e in f.get("enforce") or [] if e != "no_list"), " | ".join(str(a).strip() for a in allowed)

guideline["who_fixes"] = guideline.apply(who_fixes, axis=1)
guideline[["portal_expects", "allowed_values"]] = guideline.apply(expects, axis=1, result_type="expand")

guideline.to_csv(MAPPING / "guideline_profile.csv", index=False)
guideline.to_json(MAPPING / "guideline_profile.json", orient="records", indent=1, force_ascii=False)
print(f"Saved {(MAPPING / 'guideline_profile.csv').relative_to(REPO)} and its JSON twin")
pd.crosstab(guideline.who_fixes, guideline.usage)

Saved earthportal_sa_audit/mapping/guideline_profile.csv and its JSON twin


usage,Mandatory,Optional,Recommended
who_fixes,,,
author,12,20,54
curator,3,1,4
portal,5,0,0


**Conclusion**: `guideline_profile.csv` is the guideline in a form a program can use. For instance, the
Mandatory attributes of one record can be checked in three lines; notebook 05 does it for every ontology.

# IV. Which attributes each O'FAIRe question reads

O'FAIRe names the fields with MOD 1.4 properties and reads each one from the API by its plain name
(`reference/ofaire_properties.config.json`, taken from the O'FAIRe source code). Each O'FAIRe result
lists the properties of its question. This list comes from O'FAIRe's configuration, and the code that
gives the score reads the same properties except in a few places, kept with their evidence in
`reference/ofaire_code_vs_config.csv`. We apply them, so that every question is attached to the
cheatsheet attributes the code really reads.

In [6]:
ofaire_fields = {}
for p in json.load(open(OFAIRE_PROPERTIES, encoding="utf-8"))["properties"]:
    name = p["key"] if p["source"] == "metrics" and p.get("key") else p["portal"].split(":")[1]
    ofaire_fields.setdefault(p["mod"].replace(";", ":"), (p["source"], name))

# The properties of each question, as the O'FAIRe results list them
ofaire = json.load(open(RAW / "ofaire.json", encoding="utf-8"))
reads = {}
for result in ofaire.values():
    for principle in ("Findable", "Accessible", "Interoperable", "Reusable"):
        for block in result[principle].values():
            if isinstance(block, dict):
                for qid, q in block.get("results", {}).items():
                    reads.setdefault(qid, {"max_credits": q["maxCredits"], "properties": set()})
                    reads[qid]["properties"] |= set((q.get("properties") or {}).keys())

# We correct the lists where the code reads something else than the configuration declares
code_vs_config = pd.read_csv(CODE_VS_CONFIG)
for c in code_vs_config.itertuples():
    if c.change == "add":
        reads[c.question]["properties"].add(c.property)
    else:
        reads[c.question]["properties"].discard(c.property)

by_field = {(r.portal_object, r.api_key): r for r in guideline.itertuples()}
rows = []
for qid, info in reads.items():
    for prop in sorted(info["properties"]) or [""]:
        source, name = ofaire_fields.get(prop, ("", ""))
        g = by_field.get((source, name))
        who = g.who_fixes if g else ("tool or repository" if not prop else "portal" if source in ("metrics", "link") else "author")
        rows.append({"question": qid, "max_credits": info["max_credits"], "ofaire_property": prop,
                     "api_key": name, "cheatsheet_id": g.id if g else "", "cheatsheet_entry": g.entry if g else "",
                     "usage": g.usage if g else "", "who_fixes": who})
ofaire_vs_guideline = pd.DataFrame(rows)
ofaire_vs_guideline.to_csv(MAPPING / "ofaire_vs_guideline.csv", index=False)

not_in_cheatsheet = ofaire_vs_guideline[(ofaire_vs_guideline.ofaire_property != "") & (ofaire_vs_guideline.cheatsheet_entry == "")]
print(f"{ofaire_vs_guideline.question.nunique()} questions, {(ofaire_vs_guideline.ofaire_property != '').sum()} properties read; "
      f"{len(not_in_cheatsheet)} of them have no entry in the cheatsheet")
ofaire_vs_guideline.head(10)

61 questions, 158 properties read; 33 of them have no entry in the cheatsheet


,question,max_credits,ofaire_property,api_key,cheatsheet_id,cheatsheet_entry,usage,who_fixes
0,F1Q1,9.0,owl:ontologyIRI,URI,1.8,URI,Mandatory,author
1,F1Q2,11.0,dct:identifier,identifier,1.14,Other identifier,Optional,author
2,F1Q3,12.0,,,,,,tool or repository
3,F1Q4,9.0,owl:versionIRI,versionIRI,1.9,Version IRI,Recommended,author
4,F2Q1,16.0,dct:alternative,alternative,2.9,Alternative name,Recommended,author
5,F2Q1,16.0,dct:audience,audience,8.2,Audience,Recommended,author
6,F2Q1,16.0,dct:coverage,coverage,9.3,Coverage,Recommended,author
7,F2Q1,16.0,dct:description,description,2.2,Description,Mandatory,author
8,F2Q1,16.0,dct:publisher,publisher,5.5,Publisher,Recommended,author
9,F2Q1,16.0,dct:subject,hasDomain,2.7,Subjects,Recommended,author


**Conclusion**: most properties read by O'FAIRe have an entry in the cheatsheet. Those without one are
either computed by the portal (metrics, links) or fields the guideline chose not to describe (the object
description properties, the version information, the deprecation flag); they appear in the table above
with an empty `cheatsheet_entry`.